<a href="https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w08_warehouse_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Warehouse Features and Time-Aware Labels

My Week 5 model was trained on the 30,000-row anonymized starter CSV. Two things
about that bothered me enough to redo it:

1. The features are **contemporaneous** with the label. The starter's 90-day
   aggregates cover the same window the decline label is measured over, so the
   result is an association, not a forecast.
2. The capstone card points at the **full warehouse**, and my Week 4 data
   contract already queries it with DuckDB over `hf://`. There is no good reason
   to model on the small slice.

This notebook rebuilds the feature table from
`fact_content_daily_performance` and defines a label on a **future window**:
features come from days before an anchor date T, the outcome is measured strictly
after T.

**Section 1** surveys what the warehouse actually contains, so the windows are
chosen from the data rather than assumed.

In [ ]:
# ============================================================
# CAPSTONE — SECTION 1
# WHAT IS ACTUALLY IN THE WAREHOUSE
# ============================================================

# REASONING:
# Before choosing a feature window and an outcome window I need to know how many
# months exist, how dense each one is, and whether coverage is stable enough to
# support a future-window label.
#
# This cell reads only aggregates. No row-level data and no private fields.

%pip install -q duckdb huggingface_hub pandas

import os

import duckdb
import pandas as pd

# ------------------------------------------------------------
# TOKEN
# ------------------------------------------------------------
#
# The token is never written into this notebook. In Colab it comes from
# Secrets; locally it comes from the HF_TOKEN environment variable.

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Token source: Colab Secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("Token source: HF_TOKEN environment variable")

if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token found.\n"
        "In Colab: add HF_TOKEN under the key icon in the left sidebar.\n"
        "Locally:  set HF_TOKEN in your environment before starting Jupyter."
    )

# ------------------------------------------------------------
# CONNECT
# ------------------------------------------------------------

con = duckdb.connect()

con.execute(
    "CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

daily = (
    f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet', "
    "hive_partitioning = true)"
)

print("Connected.\n")

# ------------------------------------------------------------
# MONTH-BY-MONTH COVERAGE
# ------------------------------------------------------------

coverage = con.execute(f"""
SELECT
    month,
    COUNT(*)                            AS rows,
    COUNT(DISTINCT client_hash_id)      AS clients,
    COUNT(DISTINCT content_hash_id)     AS pages,
    MIN(report_date)                    AS min_date,
    MAX(report_date)                    AS max_date,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows
FROM {daily}
GROUP BY month
ORDER BY month
""").df()

print("WAREHOUSE COVERAGE BY MONTH")
print("=" * 78)
print(coverage.to_string(index=False))

print("\nMonths:", len(coverage))
print("Total rows:", f"{coverage['rows'].sum():,}")
print("Date span:", coverage["min_date"].min(), "to", coverage["max_date"].max())


### What I am looking for in the output above

- **How many months**, and whether the row counts are steady or ramping. A
  ramping panel means early months have thinner client coverage.
- **Whether `pages` is stable** across months. If the page population churns
  heavily, a page present in the feature window may be absent from the outcome
  window, and those rows need an explicit decision rather than a silent drop.
- **The GSC vs GA4 split.** Week 4 found GA4 available on only 413,966 of
  9,841,378 March rows, so engagement features will be sparse and must be
  imputed honestly rather than zero-filled.

The feature window and outcome window are chosen in Section 2, once these
numbers are on the page.

## Section 2 — Choosing the windows, and the label

The coverage table decides this, so the choice is recorded here with the reason.

**The panel ramps.** January 2025 has 2 clients and 476 pages. The warehouse only
becomes substantial from November 2025 (43 clients, 6.8M rows) and is stable and
dense across the last four months (61-66 clients, 362k-409k pages). Early months
are not a smaller version of the panel, they are a different one, so training on
them would mix two populations.

**GA4 is mostly absent.** It is exactly zero for the first nine months and reaches
only 644,726 of 11,694,072 rows by June 2026, about 5.5%. Engagement features are
therefore sparse across the whole warehouse and must be imputed, never zero-filled.

**The last full month is June 2026.** That makes it the outcome window, and the
90 days before it the feature window:

| Window | Dates | Used for |
|---|---|---|
| `w3` | 2026-03-03 to 2026-04-01 | feature |
| `w2` | 2026-04-02 to 2026-05-01 | feature |
| `w1` | 2026-05-02 to 2026-05-31 | feature |
| **T** | **2026-06-01** | **prediction point** |
| outcome | 2026-06-01 to 2026-06-30 | label only |

Three equal 30-day windows before T, one equal 30-day window after it.

**The label.** Mirroring the definition FlyRank documents for `trend_direction`,
a page is labelled 1 when impressions fall by more than 20% from `w1` to the
outcome window:

    label = 1 when (outcome_impressions - w1_impressions) / w1_impressions < -0.20

**Why this is not the Week 5 label.** In the starter dataset both sides of that
comparison were features, so the label could be rebuilt exactly from the feature
set. Here the numerator is measured after T and is unknown at prediction time.
The denominator, `w1`, stays a feature, because knowing a page's current traffic
level is not the same as knowing where it goes next.

That change also returns the six momentum columns I had to delete in Week 5.
Movement measured before T cannot encode an outcome measured after T, so `w1`
versus `w2` versus `w3` are legitimate features under this design.

**Pages that disappear.** A page present before T but absent from June is not
missing data, it is a page whose impressions went to zero, which is the most
severe decline there is. Dropping those rows would be survivorship bias, so the
outcome is left-joined and absence is read as zero impressions.

In [ ]:
# ============================================================
# CAPSTONE — SECTION 2
# BUILD THE FEATURE TABLE AND THE FUTURE-WINDOW LABEL
# ============================================================

# REASONING:
# Everything below aggregates the daily fact table into one row per
# client x content, using ONLY days before the prediction point T.
#
# The label is computed separately, from days on or after T, and joined on at
# the end. Keeping the two queries apart is deliberate: it makes it structurally
# hard for an outcome-window column to end up in the feature set, which is
# exactly the mistake I made in Week 5.

T           = "2026-06-01"   # prediction point

W1_START, W1_END = "2026-05-02", "2026-05-31"   # most recent 30d before T
W2_START, W2_END = "2026-04-02", "2026-05-01"   # 30d before that
W3_START, W3_END = "2026-03-03", "2026-04-01"   # 30d before that

OUT_START, OUT_END = "2026-06-01", "2026-06-30"  # outcome window, after T

FEATURE_MONTHS = "'2026-03', '2026-04', '2026-05'"
OUTCOME_MONTHS = "'2026-06'"

print("Prediction point T:", T)
print("Feature window:", W3_START, "to", W1_END)
print("Outcome window:", OUT_START, "to", OUT_END)

# ------------------------------------------------------------
# WHAT ELSE CAN I JOIN? — dim_content schema
# ------------------------------------------------------------
#
# I know the daily table's columns from Week 4. I do not want to guess
# dim_content's, so this prints them rather than assuming.

content_schema = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content/**/*.parquet') LIMIT 1
""").df()

print("\ndim_content columns:")
print(content_schema[["column_name", "column_type"]].to_string(index=False))

# ------------------------------------------------------------
# FEATURES — DAYS BEFORE T ONLY
# ------------------------------------------------------------

feature_sql = f"""
WITH pre AS (
    SELECT *
    FROM {daily}
    WHERE month IN ({FEATURE_MONTHS})
      AND CAST(report_date AS DATE) BETWEEN DATE '{W3_START}'
                                        AND DATE '{W1_END}'
)
SELECT
    client_hash_id,
    content_hash_id,

    -- 90-day totals
    SUM(gsc_impressions)                                   AS impressions_90d,
    SUM(gsc_clicks)                                        AS clicks_90d,
    SUM(ga4_sessions)                                      AS sessions_90d,
    SUM(ga4_users)                                         AS users_90d,
    SUM(ga4_engaged_sessions)                              AS engaged_sessions_90d,
    SUM(sessions_ai)                                       AS ai_sessions_90d,
    SUM(scroll_events)                                     AS scroll_events_90d,
    SUM(ga4_total_engagement_sec)                          AS engagement_sec_90d,

    -- presence
    COUNT(DISTINCT CASE WHEN gsc_impressions > 0
                        THEN report_date END)              AS days_with_impressions,
    COUNT(DISTINCT CASE WHEN ga4_sessions > 0
                        THEN report_date END)              AS days_with_sessions,

    -- position: sum/impressions is the correct weighted average
    SUM(gsc_sum_position)                                  AS sum_position_90d,

    -- momentum: three equal 30-day windows, all before T
    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W1_START}' AND DATE '{W1_END}'
             THEN gsc_impressions ELSE 0 END)              AS imp_w1,
    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W2_START}' AND DATE '{W2_END}'
             THEN gsc_impressions ELSE 0 END)              AS imp_w2,
    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W3_START}' AND DATE '{W3_END}'
             THEN gsc_impressions ELSE 0 END)              AS imp_w3,

    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W1_START}' AND DATE '{W1_END}'
             THEN gsc_clicks ELSE 0 END)                   AS clicks_w1,
    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W2_START}' AND DATE '{W2_END}'
             THEN gsc_clicks ELSE 0 END)                   AS clicks_w2,

    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W1_START}' AND DATE '{W1_END}'
             THEN ga4_sessions ELSE 0 END)                 AS sessions_w1,
    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W2_START}' AND DATE '{W2_END}'
             THEN ga4_sessions ELSE 0 END)                 AS sessions_w2,

    -- data availability, kept as honest context rather than dropped
    MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS had_gsc,
    MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS had_ga4

FROM pre
GROUP BY client_hash_id, content_hash_id
"""

print("\nBuilding features (this reads ~32M daily rows, allow a few minutes)...")
features = con.execute(feature_sql).df()
print("Feature rows:", f"{len(features):,}")

# ------------------------------------------------------------
# LABEL — DAYS ON OR AFTER T ONLY
# ------------------------------------------------------------

label_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_next_30d
FROM {daily}
WHERE month IN ({OUTCOME_MONTHS})
  AND CAST(report_date AS DATE) BETWEEN DATE '{OUT_START}' AND DATE '{OUT_END}'
GROUP BY client_hash_id, content_hash_id
"""

print("Building labels...")
outcome = con.execute(label_sql).df()
print("Outcome rows:", f"{len(outcome):,}")

# ------------------------------------------------------------
# JOIN — LEFT, SO DISAPPEARING PAGES ARE KEPT AS ZERO
# ------------------------------------------------------------

panel = features.merge(
    outcome,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

vanished = panel["impressions_next_30d"].isna().sum()
panel["impressions_next_30d"] = panel["impressions_next_30d"].fillna(0)

print("\nPages with no June rows at all (treated as zero impressions):",
      f"{vanished:,}")

# ------------------------------------------------------------
# ELIGIBILITY AND LABEL
# ------------------------------------------------------------
#
# The ratio needs a non-zero denominator. A page with no impressions in w1 has
# no level to fall from, so it is out of scope rather than labelled 0.

eligible = panel[panel["imp_w1"] > 0].copy()

print("Rows before eligibility filter:", f"{len(panel):,}")
print("Rows with imp_w1 > 0:", f"{len(eligible):,}")

eligible["impressions_change_pct"] = (
    (eligible["impressions_next_30d"] - eligible["imp_w1"])
    / eligible["imp_w1"] * 100
)

eligible["is_declining_label"] = (
    eligible["impressions_change_pct"] < -20
).astype(int)

print("\n" + "=" * 62)
print("FUTURE-WINDOW PANEL")
print("=" * 62)
print("Pages:", f"{len(eligible):,}")
print("Clients:", eligible["client_hash_id"].nunique())
print("Declining (label = 1):", f"{int(eligible['is_declining_label'].sum()):,}")
print("Base rate:", round(eligible["is_declining_label"].mean(), 4))
print("Pages that went to zero impressions:",
      f"{int((eligible['impressions_next_30d'] == 0).sum()):,}")

print("\nLabel distribution:")
display(
    eligible["is_declining_label"]
    .value_counts()
    .rename_axis("is_declining_label")
    .reset_index(name="pages")
)

print("Median impressions in w1:", eligible["imp_w1"].median())
print("GA4 available on:", f"{int(eligible['had_ga4'].sum()):,}", "of",
      f"{len(eligible):,}", "pages")


### Before modelling, check these in the output above

- **Base rate.** The starter slice was 54.2% declining. If this comes back wildly
  different, the future-window task is not the same task, and the paper has to say
  so rather than compare the two numbers as if they were.
- **How many pages went to zero.** A large share means the label is dominated by
  pages disappearing rather than declining, which is a different phenomenon and
  changes what the recommendation means.
- **GA4 coverage on eligible pages.** If it is very low, the engagement features
  carry almost no information and should be reported as such rather than listed
  as if they contributed.

Section 3 runs the leakage audit against this feature set, then trains the
baseline and the model under both a client-grouped and a time-aware split.